# 02 — Modelling

**Boréal Marché — Head A Retention Prediction**

This notebook runs the training pipeline interactively for inspection.
All modelling code lives in `src/train.py` — this notebook only calls
those functions and visualises the results.

## Contents
1. Spot-check: 8 families + 2 baselines
2. Optuna HPO: tuning the top 2 models
3. Calibration: isotonic vs sigmoid
4. Threshold selection: profit-optimal operating point
5. Final held-out evaluation
6. Error analysis by segment

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

from src.features import load_config, FEATURE_NAMES
from src.train import (
    _get_models, spot_check, tune_model, calibrate,
    select_threshold, error_analysis,
)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

## 1. Load Features

In [ ]:
from pathlib import Path

ROOT = Path("..").resolve()
X = pd.read_parquet(ROOT / "data/processed/X.parquet")
y = pd.read_parquet(ROOT / "data/processed/y.parquet").squeeze()

cfg = load_config()
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=cfg["modelling"]["test_size"],
    stratify=y,
    random_state=cfg["data"]["seed"],
)
print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")
print(f"Train class balance: {y_train.value_counts(normalize=True).to_dict()}")

## 2. Spot-Check

Compare 8 model families + 2 baselines under identical 5-fold CV.

In [ ]:
results = spot_check(X_train, y_train, cfg)
results_df = pd.DataFrame(results).sort_values("mean_pr_auc", ascending=False)
results_df

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(results_df["model"], results_df["mean_pr_auc"])
ax.set_xlabel("Mean PR-AUC (5-fold CV)")
ax.set_title("Spot-Check Results")
plt.tight_layout()
plt.show()

## 3. Optuna HPO

Tune the top 2 models with Bayesian search (100 trials each).

In [ ]:
# Tune the top model (HistGradientBoosting)
best_model, study = tune_model(
    "HistGradientBoostingClassifier", X_train, y_train, cfg
)
print(f"Best PR-AUC: {study.best_value:.4f}")
print(f"Best params: {study.best_params}")

In [ ]:
# Optuna visualisation
import optuna
optuna.visualization.matplotlib.plot_optimization_history(study)
plt.title("Optuna Optimisation History")
plt.tight_layout()
plt.show()

## 4. Calibration

Reliability curves before and after isotonic / sigmoid calibration.

In [ ]:
cal_model, cal_results = calibrate(best_model, X_train, y_train)
print("Calibration results:")
for method, brier in cal_results.items():
    print(f"  {method}: Brier = {brier:.4f}")

## 5. Threshold Selection

Expected-profit curve with cost assumptions from `pipeline.yaml`.

In [ ]:
threshold, profit_curve = select_threshold(cal_model, X_train, y_train, cfg)
print(f"Profit-optimal threshold: {threshold:.3f}")

fig, ax = plt.subplots()
ax.plot(profit_curve["thresholds"], profit_curve["profits"])
ax.axvline(threshold, color="red", ls="--", label=f"Optimal = {threshold:.3f}")
ax.set_xlabel("Threshold")
ax.set_ylabel("Expected Profit ($)")
ax.set_title("Profit Curve")
ax.legend()
plt.tight_layout()
plt.show()

## 6. Final Test-Set Evaluation

Reported once. The test set was not touched during spot-check or HPO.

In [ ]:
from sklearn.metrics import (
    average_precision_score, roc_auc_score, brier_score_loss,
    classification_report,
)

y_prob = cal_model.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= threshold).astype(int)

print(f"PR-AUC:    {average_precision_score(y_test, y_prob):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_prob):.4f}")
print(f"Brier:     {brier_score_loss(y_test, y_prob):.4f}")
print(f"Threshold: {threshold:.3f}")
print()
print(classification_report(y_test, y_pred, target_names=["churned", "returned"]))

## 7. Error Analysis

In [ ]:
errors = error_analysis(cal_model, X_test, y_test, threshold)
print("Segments where the model underperforms:")
for segment, metrics in errors.items():
    print(f"  {segment}: recall={metrics.get('recall', 'N/A'):.2f}, "
          f"precision={metrics.get('precision', 'N/A'):.2f}")